In [1]:
import sys
print(sys.executable)
!{sys.executable} -m pip install pyspedas pytplot pandas matplotlib numpy

c:\Users\madis\AppData\Local\Programs\Python\Python312\python.exe



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import matplotlib.ticker as mticker
import pyspedas
import random
import matplotlib.dates as mdates

<frozen importlib._bootstrap>:488: RuntimeWarning: numpy.ndarray size changed, may indicate binary incompatibility. Expected 16 from C header, got 96 from PyObject


In [3]:
# Choose a folder on your local SSD for caching
# Fully normalized absolute path
cache_dir = os.path.abspath(os.path.join(os.path.expanduser('~'), 'pyspedas_cache'))
os.makedirs(cache_dir, exist_ok=True)

pyspedas.data_dir = cache_dir
print(f"PySPEDAS cache directory: {pyspedas.data_dir}")

from pyspedas.projects.omni import load
from pyspedas.projects import themis
from pyspedas import get_data

PySPEDAS cache directory: C:\Users\madis\pyspedas_cache


In [ ]:
# Define Events
event_tha = pd.read_csv('event_data_tha_perp_80_avg.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'event_data_tha_perp_80_avg.csv'

In [ ]:
# Filter the CSV for only satellite THA
event_tha_only = event_tha[event_tha['satellite_name'] == 'THEMIS A']

print(event_tha_only['satellite_name'].unique())
print(len(event_tha), len(event_tha_only))

In [ ]:
# Defined time range
start = event_tha_only['start_time'].min()
end = event_tha_only['end_time'].max()

In [ ]:
# Magnetic field (Bz)
themis.fgm(trange=[start, end], probe='a', level='l2', coord='gsm', get_support_data=False)

In [ ]:
# Step 1: Build Matrix M with Magnetic field Base
fgm_var = get_data('tha_fgs_gsm')

times_fgm = fgm_var[0]
B = fgm_var[1]

GSM_matrix = pd.DataFrame({
    'time': pd.to_datetime(times_fgm, unit='s'),
    'Bx': B[:,0],
    'By': B[:,1],
    'Bz': B[:,2]
})

In [ ]:
# Map each event's start and end time in event_tha to index ranges in matrix time using UNIX times
tha_start_time = (pd.to_datetime(event_tha_only['start_time']).astype('int64') // 10**9).values
tha_end_time   = (pd.to_datetime(event_tha_only['end_time']).astype('int64') // 10**9).values

import numpy as np

event_indices = []

for idx, row in event_tha_only.iterrows():
    start_idx = np.searchsorted(GSM_matrix['Epoch_time'], tha_start_time[idx], side='left')
    end_idx   = np.searchsorted(GSM_matrix['Epoch_time'], tha_end_time[idx], side='right')

    event_indices.append(GSM_matrix.iloc[start_idx:end_idx].index.values)

In [ ]:
random.seed(18)
event_sample = random.sample(range(len(event_indices)), 10)

for i in event_sample:
    idx_range = event_indices[i]

    if len(idx_range) == 0:
        continue

    rand_event_df = GSM_matrix.loc[idx_range[0]:idx_range[-1]]

    plt.figure(figsize=(10, 6))
    plt.plot(rand_event_df['Time'], rand_event_df['Bx'], label='Bx')
    plt.plot(rand_event_df['Time'], rand_event_df['By'], label='By')
    plt.plot(rand_event_df['Time'], rand_event_df['Bz'], label='Bz')

    plt.xlabel('Time')
    plt.ylabel('Magnetic Field (nT)')
    plt.legend()

    plt.title(
        f"Event {i+1} | Index {idx_range[0]}–{idx_range[-1]} | "
        f"Class: {rand_event_df['Event_class'].max()}"
    )

    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
    plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.xticks(rotation=45)

    plt.grid()
    plt.show()